# Integration Fundamentals

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 02.06 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/07_integration_fundamentals.ipynb)

---

## 🎯 Learning Objective

Connect the definite integral to area accumulation via Riemann sums, and state the fundamental theorem of calculus linking integration and differentiation.

---

## 📐 Theory

Differentiation measures instantaneous change; integration reverses direction and asks: given
a rate of change, how much accumulates in total? The two turn out to be inverse operations —
that fact alone is the deepest result in this notebook.

### Riemann sums

The **definite integral** $\int_a^b f(x)\,dx$ is defined as the area under $f$ between $a$ and
$b$, built up as a limit of rectangle approximations. Partition $[a,b]$ into $n$ strips of
width $\Delta x = (b-a)/n$, and sum the area of each rectangle:

$$\int_a^b f(x)\,dx = \lim_{n\to\infty}\sum_{i=1}^n f(x_i^*)\,\Delta x$$

where $x_i^*$ is any point in the $i$-th strip. As $n\to\infty$ and $\Delta x\to 0$, the jagged
rectangle approximation converges to the exact area — positive where $f>0$, negative where
$f<0$ (integration counts *signed* area).

### The Fundamental Theorem of Calculus

This is the result that makes integration computable without summing infinitely many
rectangles. Define $F(x) = \int_a^x f(t)\,dt$ (the "accumulated area so far" function). The
theorem states:

$$F'(x) = f(x) \qquad\text{and}\qquad \int_a^b f(x)\,dx = F(b) - F(a)$$

Part one says accumulation and differentiation *undo each other*: the derivative of "area
accumulated up to $x$" is just the height of the function at $x$. Part two says you can compute
any definite integral by finding *any* antiderivative $F$ (a function whose derivative is $f$)
and evaluating $F(b)-F(a)$ — no limits of sums required. An **indefinite integral**
$\int f(x)\,dx = F(x) + C$ is the family of all antiderivatives (the constant $C$ reflects that
shifting a function vertically doesn't change its slope).

### Basic integration rules

Integration inherits the reverse of every differentiation rule from `02.02`:
$\int x^n\,dx = \frac{x^{n+1}}{n+1}+C$ (for $n\ne -1$, the reverse power rule), integrals
distribute over sums, and constants pull outside: $\int c\,f(x)\,dx = c\int f(x)\,dx$. There's
no general product or quotient rule for integrals (unlike differentiation) — composite
integrals often need substitution or the techniques covered in `02.08`.

### Why integration matters here

An integral is fundamentally an **accumulation** or **averaging** operation, and both ideas
recur constantly once probability enters the picture (Module `03`): a probability density's
total area must equal $1$ (it's a normalized accumulation), and an expected value
$\mathbb{E}[X] = \int x\, p(x)\,dx$ is literally a weighted integral. Anywhere you see
"area under the curve," "cumulative," or "expected value" in the notebooks ahead, an integral
is doing the work underneath.

---

In [ ]:
# Setup
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Rectangle approximations of increasing fineness converge visually to the exact area under a
curve, making the Riemann sum limit concrete rather than abstract.

In [ ]:
# Riemann sums with increasing n converging to the exact area under f(x) = sin(x) + 1.5 on [0, pi]
f = lambda x: np.sin(x) + 1.5
a, b = 0, np.pi
exact_area = -np.cos(b) + np.cos(a) + 1.5 * (b - a)  # antiderivative: -cos(x) + 1.5x

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, n in zip(axes, [4, 16, 64]):
    x_fine = np.linspace(a, b, 400)
    ax.plot(x_fine, f(x_fine), color="#4C72B0", lw=2)
    edges = np.linspace(a, b, n + 1)
    width = (b - a) / n
    midpoints = (edges[:-1] + edges[1:]) / 2
    heights = f(midpoints)
    riemann_sum = np.sum(heights * width)
    ax.bar(midpoints, heights, width=width, alpha=0.4, edgecolor="#DD8452", color="#DD8452")
    ax.set_title(f"n={n} rectangles\nsum={riemann_sum:.4f} (exact={exact_area:.4f})")
    ax.set_xlabel("x"); ax.set_ylabel("f(x)")
plt.tight_layout()
plt.show()

print(f"Exact integral (fundamental theorem): {exact_area:.6f}")
for n in [4, 16, 64, 256, 1024]:
    edges = np.linspace(a, b, n + 1)
    midpoints = (edges[:-1] + edges[1:]) / 2
    riemann_sum = np.sum(f(midpoints) * (b - a) / n)
    print(f"n={n:5d}: Riemann sum = {riemann_sum:.6f}  (error = {abs(riemann_sum - exact_area):.2e})")

## 🐍 Implementation from Scratch

We implement the trapezoidal rule (a refinement of the Riemann sum idea that connects sample
points with straight lines instead of flat rectangle tops, converging faster), then verify the
Fundamental Theorem of Calculus itself: numerically differentiate an accumulated-area function
and confirm it reproduces the original integrand.

In [ ]:
def trapezoidal_rule(f, a, b, n=1000):
    """Trapezoidal Riemann sum: average the LEFT and RIGHT edge heights of each strip
    instead of using one sample point -- this cancels out first-order error and converges faster."""
    x = np.linspace(a, b, n + 1)
    y = f(x)
    widths = np.diff(x)
    return np.sum(widths * (y[:-1] + y[1:]) / 2)

f = lambda x: np.sin(x) + 1.5
approx = trapezoidal_rule(f, 0, np.pi, n=1000)
exact = -np.cos(np.pi) + np.cos(0) + 1.5 * np.pi
print(f"Trapezoidal approximation: {approx:.6f}")
print(f"Exact (fundamental theorem): {exact:.6f}")
print(f"Cross-check with SymPy: {float(sp.integrate(sp.sin(sp.Symbol('x')) + 1.5, (sp.Symbol('x'), 0, sp.pi))):.6f}")

# --- Verify the Fundamental Theorem of Calculus itself: F'(x) should equal f(x) ---
def accumulated_area(f, a, x_val, n=2000):
    # F(x) = integral from a to x -- build it as a callable "area so far" function
    return trapezoidal_rule(f, a, x_val, n=n)

def numeric_derivative(F, x, h=1e-4):
    return (F(x + h) - F(x - h)) / (2 * h)

test_points = [0.5, 1.5, 2.5]
print("\nFundamental Theorem check: is F'(x) == f(x)?")
for x_val in test_points:
    F_prime = numeric_derivative(lambda t: accumulated_area(f, 0, t), x_val)
    print(f"  x={x_val}: F'(x) = {F_prime:.5f},  f(x) = {f(x_val):.5f},  match = {np.isclose(F_prime, f(x_val), atol=1e-3)}")

## 🤖 Why This Matters for AI

Every probability density function (Module `03.02` onward) is defined so that
$\int_{-\infty}^{\infty} p(x)\,dx = 1$ — a normalized area. Expected value,
$\mathbb{E}[X] = \int x\,p(x)\,dx$, is a weighted integral: exactly the calculation behind
average loss over a data distribution. And a metric you've likely seen reported for any
classifier, **AUC-ROC** (Area Under the ROC Curve), is a literal Riemann-sum-style integral of
the true-positive rate against the false-positive rate as a decision threshold sweeps from $0$
to $1$ — no metaphor, an actual numerical integral computed the same way we did above. Below,
we compute AUC-ROC for a toy classifier's predictions two ways: via `sklearn`'s built-in
function, and via our own `trapezoidal_rule`, confirming they agree.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

# A toy binary classifier's predicted scores vs. true labels
rng = np.random.default_rng(0)
n = 300
y_true = rng.integers(0, 2, n)
# Scores that are correlated with the true label but noisy -- a realistic imperfect classifier
y_score = y_true * 0.6 + rng.normal(scale=0.4, size=n)

fpr, tpr, thresholds = roc_curve(y_true, y_score)   # sklearn gives us the (FPR, TPR) curve points

# AUC-ROC IS the area under this curve -- compute it with our own trapezoidal_rule by
# treating (fpr, tpr) as (x, y) samples of a function, sorted by x
order = np.argsort(fpr)
fpr_sorted, tpr_sorted = fpr[order], tpr[order]
auc_manual = np.sum(np.diff(fpr_sorted) * (tpr_sorted[:-1] + tpr_sorted[1:]) / 2)  # trapezoidal, directly

auc_sklearn = roc_auc_score(y_true, y_score)
print(f"AUC-ROC via sklearn's roc_auc_score: {auc_sklearn:.5f}")
print(f"AUC-ROC via our own trapezoidal integration of the ROC curve: {auc_manual:.5f}")
print(f"Match: {np.isclose(auc_sklearn, auc_manual, atol=1e-3)}")

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot(fpr, tpr, color="#4C72B0", lw=2, label=f"ROC curve (AUC={auc_sklearn:.3f})")
ax.fill_between(fpr, tpr, alpha=0.2, color="#4C72B0")
ax.plot([0, 1], [0, 1], 'k--', lw=1, label="random classifier (AUC=0.5)")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("AUC-ROC is a literal integral: the shaded area under this curve")
ax.legend()
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Compute $\int_0^2 (3x^2 - 4x + 1)\,dx$ by hand using the fundamental theorem, then verify
   with `trapezoidal_rule` and `sp.integrate`. The integrand changes sign inside $[0,2]$ (at
   $x=\frac13$ and $x=1$) — split the integral into the three pieces $[0,\frac13]$,
   $[\frac13,1]$, $[1,2]$, confirm each piece's sign matches whether $f$ is positive or negative
   on that sub-interval, and confirm the three pieces still sum to your original answer.

<details>
<summary>💡 Solution</summary>

An antiderivative is $F(x)=x^3-2x^2+x$ (reverse power rule, term by term), so
$\int_0^2(3x^2-4x+1)\,dx = F(2)-F(0) = (8-8+2)-0=2$. Both `trapezoidal_rule(f, 0, 2, n=1000)`
and `sp.integrate` reproduce $2.0$. Since $3x^2-4x+1=(3x-1)(x-1)$, $f$ is positive on
$[0,\frac13)$, negative on $(\frac13,1)$, and positive again on $(1,2]$. Computing each piece:
$\int_0^{1/3}f\,dx=\frac{4}{27}\approx0.1481$ (positive, matching $f>0$ there),
$\int_{1/3}^1 f\,dx=-\frac{4}{27}\approx-0.1481$ (negative, matching $f<0$ there), and
$\int_1^2 f\,dx=2$ (positive, matching $f>0$ there). Summing:
$\frac{4}{27}-\frac{4}{27}+2=2$ — identical to the direct computation, confirming that splitting
a signed integral at its sign changes and re-summing the pieces always recovers the same total,
with each piece's own sign tracking the integrand's sign on that sub-interval exactly.

</details>

### 💭 UNDERSTAND
2. A particle moves along a line with velocity $v(t)=t-2$ (in m/s) for $t\in[0,4]$ seconds. A
   learner computes $\int_0^4 v(t)\,dt = 0$ and concludes "the particle never moved, since its
   integral is zero." Explain what's wrong with this conclusion, compute the quantity that
   *does* correctly describe "how far the particle actually traveled," and show the two
   quantities are different.

<details>
<summary>💡 Solution</summary>

$\int_0^4 v(t)\,dt$ is the particle's **net displacement** — a signed accumulation, where
motion in the negative direction ($t<2$) cancels motion in the positive direction ($t>2$). It
correctly equals $0$ here because the particle moves backward for the first half of the trip and
forward by the exact same amount over the second half, ending up back where it started. That is
a true and useful fact, but it isn't the same question as "how far did the particle travel"
(an odometer reading), which needs $\int_0^4 |v(t)|\,dt$ — an **unsigned** accumulation, since
distance traveled keeps adding up regardless of direction. Here $|v(t)|=|t-2|$, and
$\int_0^4|t-2|\,dt = \int_0^2(2-t)\,dt+\int_2^4(t-2)\,dt = 2+2=4$ meters — nonzero, unlike the
displacement. Both `trapezoidal_rule(v, 0, 4)` (giving $\approx0$) and
`trapezoidal_rule(lambda t: np.abs(v(t)), 0, 4)` (giving $4.0$) confirm this numerically: the
integral of $v$ answers "net change in position," the integral of $|v|$ answers "total
distance," and conflating the two is exactly the mistake of assuming a *signed* accumulation
answers an *unsigned* question.

</details>

### ✏️ DERIVE
3. Derive $\int_0^3 x^2\,dx$ from the definition, using right-endpoint Riemann sums: partition
   $[0,3]$ into $n$ strips of width $\Delta x=3/n$, write the sum $\sum_{i=1}^n f(x_i)\,\Delta x$
   for $x_i=3i/n$, use the identity $\sum_{i=1}^n i^2 = \frac{n(n+1)(2n+1)}{6}$ to get a closed
   form in $n$, and take the limit as $n\to\infty$. Confirm your limit matches the
   fundamental-theorem answer.

<details>
<summary>💡 Solution outline</summary>

$$\sum_{i=1}^n f(x_i)\,\Delta x=\sum_{i=1}^n\left(\frac{3i}{n}\right)^2\frac{3}{n}
=\frac{27}{n^3}\sum_{i=1}^n i^2=\frac{27}{n^3}\cdot\frac{n(n+1)(2n+1)}{6}.$$
Expanding: $\dfrac{27}{6n^3}\big(2n^3+3n^2+n\big) = 9 + \dfrac{13.5}{n} + \dfrac{4.5}{n^2}$. As
$n\to\infty$, the last two terms vanish, leaving $\lim_{n\to\infty}(\text{Riemann sum})=9$. By
the fundamental theorem, $\int_0^3x^2\,dx=\left[\frac{x^3}{3}\right]_0^3=\frac{27}{3}=9$ —
identical. Numerically, the finite-$n$ sums confirm the approach to this limit from above (as
expected for right endpoints on an increasing function): $n=10$ gives $10.395$, $n=200$ gives
$9.0676$, $n=5000$ gives $9.0027$, converging to $9$ as $n$ grows — matching both the
closed-form limit and the fundamental theorem's shortcut.

</details>

### 🐍 IMPLEMENT
4. Implement `simpsons_rule(f, a, b, n)` (even $n$): instead of `trapezoidal_rule`'s straight
   lines, fit a parabola through each successive triple of points, using the composite Simpson's
   rule $\int_a^b f\,dx\approx\frac{h}{3}\big[f(x_0)+4\sum_{\text{odd }i}f(x_i)+2\sum_{\text{even
   }i,\,i\ne0,n}f(x_i)+f(x_n)\big]$ with $h=(b-a)/n$. On $\int_0^1 e^{-x^2}\,dx$ (compare against
   `scipy.integrate.quad`), measure both methods' error at $n=8$ and $n=16$ and check how each
   method's error scales when $n$ doubles.

<details>
<summary>✅ How to know you're right</summary>

Against `scipy.integrate.quad`'s reference value ($\approx0.74682413$), `trapezoidal_rule`'s
error should shrink by a factor of about $4$ ($=2^2$) every time $n$ doubles — e.g.
$\approx9.6\times10^{-4}$ at $n=8$ dropping to $\approx2.4\times10^{-4}$ at $n=16$ — confirming
trapezoidal error is $O(h^2)$. `simpsons_rule`'s error should shrink by a factor of about $16$
($=2^4$) per doubling — e.g. $\approx2.0\times10^{-6}$ at $n=8$ dropping to
$\approx1.25\times10^{-7}$ at $n=16$ — confirming Simpson's error is the faster $O(h^4)$, and at
every common $n$ tested Simpson's error should be several orders of magnitude smaller than
trapezoidal's. If your Simpson error doesn't shrink measurably faster than trapezoidal's as $n$
grows, the most likely bug is the odd/even coefficient pattern ($4$ for odd-indexed points, $2$
for even-indexed interior points) being swapped or off by one index.

</details>

### 🤖 APPLY
5. A serving system models the waiting time (in seconds) for a request as an exponential
   distribution with rate $\lambda=1.5$, pdf $p(x)=\lambda e^{-\lambda x}$ for $x\ge0$. Using
   only `trapezoidal_rule` (integrating out to some large cutoff you choose as a stand-in for
   $\infty$), confirm $\int_0^\infty p(x)\,dx\approx1$, then compute the expected waiting time
   $\mathbb{E}[X]=\int_0^\infty x\,p(x)\,dx$ numerically and compare it against the exponential
   distribution's known closed form $\mathbb{E}[X]=1/\lambda$.

<details>
<summary>✅ What you should observe</summary>

With a cutoff large enough that $e^{-\lambda\cdot\text{cutoff}}$ is negligible (e.g. cutoff
$=30$ for $\lambda=1.5$), `trapezoidal_rule` applied to $p(x)$ alone should give a total
probability within about $10^{-5}$ of exactly $1$ — the normalization check this notebook's own
"Why This Matters for AI" section calls out for any density. The numeric expected value should
match $1/\lambda=0.6\overline{6}$ to within about $10^{-5}$ as well (in one run: $0.666666$
numeric vs. $0.666667$ closed-form). As a further check, $\mathbb{E}[X^2]-\mathbb{E}[X]^2$
computed the same numeric way should be non-negative and close to the known closed form
$1/\lambda^2=0.4\overline{4}$ — a variance, so it can never come out negative regardless of
which distribution or cutoff you try.

</details>

### 🚀 CHALLENGE
6. Using your own trapezoidal-integration-of-the-ROC-curve code from the AI section, sweep the
   noise scale in `y_score`'s generation across at least five values spanning "nearly perfect"
   to "very noisy" (e.g. $0.1, 0.4, 1.5, 5, 20$), recording AUC at each. Then re-run just the
   noisiest setting with many more samples than $n=300$. Explain, from the ROC curve's own
   shape, why AUC should approach exactly $0.5$ for a classifier whose score carries zero
   information about the label — and why your $n=300$ sweep might plateau *above* $0.5$ rather
   than reaching it.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer reports a monotonically decreasing AUC as noise scale grows (in one run:
$0.9998\to0.8881\to0.6854\to0.6156\to0.5913$ for scales $0.1,0.4,1.5,5,20$), correctly notices
it appears to be leveling off *above* $0.5$ rather than continuing to fall, and explains why: a
genuinely uninformative classifier's ROC curve is only *exactly* the diagonal
($\text{TPR}=\text{FPR}$ at every threshold, giving $\text{AUC}=0.5$) in expectation over
infinitely many samples — with only $n=300$ finite samples, random fluctuation almost always
pushes the *empirical* ROC curve slightly off the diagonal in one direction, and since AUC
collapses that whole curve to a single number, the fluctuation biases the observed value away
from exactly $0.5$ more often than it happens to cancel out within one run. A strong answer
verifies this directly by re-running the noisiest setting with many more samples (e.g.
$n=20000$): the AUC should move measurably closer to $0.5$ (in one such run, $0.5836$ at
$n=300$ dropped to $\approx0.497$ at $n=20000$, both using the same huge noise scale) —
demonstrating that the gap from $0.5$ at small $n$ was sampling noise, not a real floor, and
connecting to why an AUC estimated on a small validation/test set should always be reported with
some notion of uncertainty (e.g. a confidence interval) rather than as an exact number.

</details>

---

## 📚 Further Reading
- Stewart, *Calculus: Early Transcendentals*, Ch. 5 (Integrals)
- Fawcett, ["An Introduction to ROC Analysis"](https://www.sciencedirect.com/science/article/pii/S016786550500303X)

---

*Next notebook: [Multivariable Calculus](08_multivariable_calculus.ipynb)*